# Queries

In [1]:
import json
from datasets import load_dataset


nq = load_dataset(
    "natural_questions",
    split="train[:1000]"   # only 1000 queries
)


Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/307373 [00:00<?, ? examples/s]

KeyboardInterrupt: 

In [1]:
# queries = [item["question"]["text"] for item in nq]


queries_and_answers = [
    {
        "query": item["question"]["text"],
        "answer": item.get("answer", "").strip()
    }
    for item in nq
]


print(len(queries_and_answers))
print(queries_and_answers[:5])


NameError: name 'nq' is not defined

In [ ]:

# Save to JSON file
with open("queries_with_ans.json", "w", encoding="utf-8") as f:
    json.dump(
        queries_and_answers,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Saved {len(queries_and_answers)} query+answer pairs to queries_and_answers_1000.json")

In [3]:
from datasets import load_dataset
import json

# Stream dataset (no full download)
nq = load_dataset("natural_questions", split="train", streaming=True)

queries_and_answers = []

for i, item in enumerate(nq):
    queries_and_answers.append({
        "query": item["question"]["text"],
        "answer": item["answer"]["text"]
        # "answer": item.get("answer_text", "").strip()
    })
    
    if i >= 999:   # stop at 1000
        break

# Save
with open("queries_and_answers_1000.json", "w", encoding="utf-8") as f:
    json.dump(queries_and_answers, f, ensure_ascii=False, indent=2)

print("Saved 1000 query+answer pairs")

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

KeyError: 'answer'

In [6]:
from datasets import load_dataset
import json

nq = load_dataset(
    "natural_questions",
    split="train",
    streaming=True
)

queries_and_answers = []

for item in nq:
    query = item["question"]["text"]

    short_answers = item["annotations"]["short_answers"]

    if short_answers and len(short_answers) > 0:
        tokens = short_answers[0].get("text", [])
        answer = " ".join(tokens)   # 🔥 FIX HERE
    else:
        continue

    queries_and_answers.append({
        "query": query,
        "answer": answer.strip()
    })

    if len(queries_and_answers) >= 1000:
        break
    
print(queries_and_answers[:5])
    
with open("queries_and_answers_1000.json", "w", encoding="utf-8") as f:
    json.dump(queries_and_answers, f, ensure_ascii=False, indent=2)

print(f"Saved {len(queries_and_answers)} query+answer pairs")

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/287 [00:00<?, ?it/s]

[{'query': 'when is the last episode of season 8 of the walking dead', 'answer': 'March 18, 2018'}, {'query': 'in greek mythology who was the goddess of spring growth', 'answer': 'Persephone (/pərˈsɛfəni/; Greek: Περσεφόνη), also called Kore (/ˈkɔːriː/; "the maiden")'}, {'query': 'benefits of colonial life for single celled organisms', 'answer': ''}, {'query': 'how many season of the man in the high castle', 'answer': ''}, {'query': 'who was the first ministry head of state in nigeria', 'answer': ''}]
Saved 1000 query+answer pairs


# MS MARCO

In [1]:
from datasets import load_dataset

dataset = load_dataset("microsoft/ms_marco", "v1.1", split="train[:100]")

counts = {}

for sample in dataset:
    labels = sample["passages"]["is_selected"]
    num_relevant = sum(labels)
    counts[num_relevant] = counts.get(num_relevant, 0) + 1

print(counts)

{1: 87, 0: 2, 2: 9, 3: 2}


In [3]:
import json
from datasets import load_dataset

# Load dataset (change size if needed)
dataset = load_dataset("microsoft/ms_marco", "v1.1", split="train[:15000]")

output = []

for sample in dataset:
    query = sample["query"]
    passages = sample["passages"]["passage_text"]
    labels = sample["passages"]["is_selected"]

    # Extract ALL relevant passages
    relevant_passages = " ".join([
        p for p, l in zip(passages, labels) if l == 1
    ])

    # Skip if no relevant passage
    if not relevant_passages:
        continue

    output.append({
        "query": query,
        "answer": relevant_passages
    })

# Save to JSON
with open("queries_marco.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"Saved {len(output)} queries to queries_marco.json")

Saved 14533 queries to queries_marco.json
